In [13]:
# pdf --> chunks? stregry

In [10]:
DOCS = [
    "Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.",
    "Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.",
    "Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.",
    "Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.",
    "Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.",
]


# at a high level 2010



In [11]:
# key word
query_a = "gold loan close penalty"
query_word = set(query_a.lower().split())
query_word

{'close', 'gold', 'loan', 'penalty'}

In [12]:
word_in_chunk_1 = set(DOCS[0].lower().split())
word_in_chunk_1.intersection(query_word)

{'gold', 'loan', 'penalty'}

In [15]:
result = []
for i in DOCS:
    doc_word = set(i.lower().split())
    overlap = doc_word.intersection(query_word)
    score = len(overlap)
    result.append({
        "score": score,
        "text": i,
    })




    
    


In [16]:
# sort it based on score
result.sort(key=lambda x: x["score"], reverse=True)
result

[{'score': 3,
  'text': 'Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.'},
 {'score': 2,
  'text': 'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.'},
 {'score': 1,
  'text': 'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.'},
 {'score': 1,
  'text': 'Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.'},
 {'score': 1,
  'text': 'Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.'}]

In [23]:
def keyword_search(query, top_k=3):
    query_word = set(query.lower().split())
    result = []
    for i in DOCS:
        doc_word = set(i.lower().split())
        overlap = doc_word.intersection(query_word)
        score = len(overlap)
        result.append({
            "score": score,
            "text": i,
        })
    result.sort(key=lambda x: x["score"], reverse=True)
    # make sure score should be more then 1
    result = [i for i in result if i["score"] > 1]
    return result[:top_k]

In [24]:
# ── Test A: Customer uses exact document vocabulary ───────────────
query_a = "gold loan foreclosure penalty"
print(f'Query A: "{query_a}"')
print(f"(Words that appear exactly in the policy document)\n")


keyword_search(query_a)

Query A: "gold loan foreclosure penalty"
(Words that appear exactly in the policy document)



[{'score': 4,
  'text': 'Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.'},
 {'score': 2,
  'text': 'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.'}]

In [25]:
# ── Test B: Customer uses their own words (not the document's words) ─

query_b = "What  capital  India"
print(f'Query B: "{query_b}"')
print(f"Expected top result: gold_foreclosure")
print(f"(But 'close' and 'early' don't appear in that document)\n")

keyword_search(query_b)

Query B: "What  capital  India"
Expected top result: gold_foreclosure
(But 'close' and 'early' don't appear in that document)



[]

In [29]:
# text preprocessing = Stopword 

DOCS = [
    "Gold loan foreclosure attracts two percent penalty if closed within six months",
    "Personal loan EMI is auto debited on the fifth of every month",
    "CIBIL score below six fifty makes customer ineligible for new personal loan",
    "Loan renewal available for Tier one Tier two city customers with score above seven hundred",
    "Gold loan renewal max one lakh salaried fifty thousand self employed",
]

def tokenize(text):
    return text.lower().split()


all_words = set()
for i in DOCS:
    word = tokenize(i)
    for w in word:
        all_words.add(w)
VOCAB = list(all_words)



In [37]:
VOCAB

['emi',
 'one',
 'lakh',
 'foreclosure',
 'customer',
 'every',
 'renewal',
 'month',
 'six',
 'max',
 'two',
 'the',
 'with',
 'auto',
 'ineligible',
 'gold',
 'of',
 'personal',
 'for',
 'available',
 'attracts',
 'seven',
 'hundred',
 'new',
 'tier',
 'customers',
 'cibil',
 'below',
 'city',
 'percent',
 'fifty',
 'employed',
 'penalty',
 'closed',
 'months',
 'loan',
 'makes',
 'if',
 'debited',
 'is',
 'salaried',
 'within',
 'score',
 'fifth',
 'thousand',
 'above',
 'on',
 'self']

In [68]:
text = 'Hi I emi to close my loan emi'
tokens = tokenize(text)
tokens
# resultv = []
# for w in VOCAB:
#     if w in tokens:
#         resultv.append(w)
# resultv
result = tokens.count('hi')
result

1

In [70]:

# for i in VOCAB:
#     count = tokens.count(i)
#     print(i,count)

In [48]:
def bow_vector(text):
    tokens = tokenize(text)
    vector = []
    for w in VOCAB:
        count = tokens.count(w)
        vector.append(count)
    return vector

v1 = bow_vector("Gold loan foreclosure emi attracts two percent penalty if  emi closed within six months")

In [56]:
DOCS_VECTOR = []

for i in DOCS:
    vector = bow_vector(i)
    DOCS_VECTOR.append(vector)

print(DOCS_VECTOR[2])

[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]


In [75]:
# print(DOCS[0])
import math

def search(query, top_k=3):
    query_vector = bow_vector(query)
    result = []
    for i in range(len(DOCS_VECTOR)):
        doct_vector = DOCS_VECTOR[i]
        total = 0
        for j in range(len(query_vector)):
            diff = query_vector[j] - doct_vector[j]
            total = total + (diff * diff)

        distance = math.sqrt(total)
        result.append((distance,i))
    result.sort()
    return result
        
    

In [79]:
# VOCAB
query = "emi"
print("Query:", query)
search(query)

Query: emi


[(3.3166247903554, 1),
 (3.4641016151377544, 4),
 (3.605551275463989, 0),
 (3.605551275463989, 2),
 (4.242640687119285, 3)]

In [78]:
# v1

DOCS

['Gold loan foreclosure attracts two percent penalty if closed within six months',
 'Personal loan EMI is auto debited on the fifth of every month',
 'CIBIL score below six fifty makes customer ineligible for new personal loan',
 'Loan renewal available for Tier one Tier two city customers with score above seven hundred',
 'Gold loan renewal max one lakh salaried fifty thousand self employed']

In [ ]:
import math
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer()


DOCS = [
    "Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.",
    "Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.",
    "Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.",
    "Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.",
    "Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.",
]

doc_mat = tfidf.fit_transform(DOCS)

In [3]:
doc_mat

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 82 stored elements and shape (5, 62)>

In [17]:
# vocab = tfidf.vocabulary_
# vocab

In [18]:
# doc_mat[1].toarray()

In [23]:
def tfidf_search(query,top_k=3):
    query_vector = tfidf.transform([query])
    sims = cosine_similarity(doc_mat,query_vector)


    result = []
    for i in range(len(sims)):
        result.append((sims[i],DOCS[i]))
    
    return sorted(result,reverse=True)[:top_k]



tfidf_search("Hi how  you")  

[(array([0.]),
  'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.'),
 (array([0.]),
  'Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.'),
 (array([0.]),
  'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.')]